In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

response = llm.invoke("Hello, how are you?")

response.text

"Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"

In [4]:
response = llm.invoke("Como es el clima en Apizaco Tlaxcala?")
response.text

'El clima en Apizaco, Tlaxcala, es generalmente templado, con una temperatura media anual que oscila entre los 12 y 18 grados Celsius. La región presenta una temporada de lluvias que va de mayo a octubre, siendo los meses de junio y julio los más lluviosos. Durante el invierno, las temperaturas pueden descender, especialmente en la noche, y es común que se presenten heladas. En general, el clima es agradable, con días soleados y noches frescas. Sin embargo, es recomendable consultar un pronóstico del tiempo actualizado para obtener información más precisa y específica.'

In [5]:
response = llm.invoke("Dime que productos ofrece la tienda")
response.text

'Para poder ayudarte mejor, necesitaría saber a qué tienda te refieres. Hay muchas tiendas que ofrecen una amplia variedad de productos, desde ropa y electrónica hasta alimentos y artículos para el hogar. Si me das más detalles sobre la tienda específica, podré ofrecerte información más precisa sobre los productos que ofrece.'

In [6]:
system_prompt = """
You are a helpful assistant that help customers to find products they need.

The products are:
- Laptop
- Mouse
- Keyboard
- Monitor
- Headphones
- Speakers

"""
message = [
    ("system", system_prompt),
    ("user", "Tell me about the products")
]
response = llm.invoke(message)
response.text


"Sure! Here’s a brief overview of each product:\n\n1. **Laptop**: A portable computer that integrates all the components of a desktop computer into a single unit. Laptops are designed for mobile use and typically feature a built-in screen, keyboard, and trackpad. They come in various sizes and specifications, catering to different needs such as gaming, business, or casual use.\n\n2. **Mouse**: A pointing device that allows users to interact with a computer's graphical user interface. Mice can be wired or wireless and come in various designs, including ergonomic shapes for comfort. They typically have buttons for clicking and a scroll wheel for easy navigation.\n\n3. **Keyboard**: An input device that uses a set of keys or buttons to input data into a computer. Keyboards can be mechanical or membrane, and they come in various layouts and designs, including ergonomic options. Some keyboards also feature backlighting and programmable keys for enhanced functionality.\n\n4. **Monitor**: A s

In [7]:
from langchain_core.tools import tool
import requests 

@tool("get_products", description="Get the products available in the store filter by price")
def get_products():
    ## Connect with API
    """Get the products available in the store"""
    
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    
    return "".join([f"{product['title']} - {product['price']}\n" for product in products])

In [8]:
get_products.invoke({})

'Test Product 71e5817b-666e-439f-962a-475c8c4e9c2a - 100\nProduct 1 - Demo - 47\nProduct 5 - Demo - 195\nProduct 4 - Demo - 158\nProduct 3 - Demo - 121\nProduct 2 - Demo - 84\nProduct 9 - Demo - 343\nProduct 6 - Demo - 232\nProduct 7 - Demo - 269\nProduct 8 - Demo - 306\nProduct 12 - Demo - 454\nProduct 11 - Demo - 417\nProduct 13 - Demo - 491\nProduct 15 - Demo - 65\nProduct 14 - Demo - 28\nProduct 18 - Demo - 176\nProduct 20 - Demo - 250\nProduct 16 - Demo - 102\nProduct 17 - Demo - 139\nProduct 19 - Demo - 213\nProduct 21 - Demo - 287\nProduct 24 - Demo - 398\nProduct 25 - Demo - 435\nProduct 23 - Demo - 361\nProduct 22 - Demo - 324\nTest Automation Products - 81\ntestando - 111\nteste produto - 111\nteste 12131 - 214124\nteste 123124124 - 111111\nTest Automation Product1774703642590 - 81\nTest Automation Product1774703656755 - 81\nafafafa - 1231231\nUnbranded Marble Cheese - 603\nSleek Metal Chicken - 318\nElegant Metal Soap - 702\nGorgeous Silk Fish - 315\nErgonomic Steel Car - 36

In [9]:
@tool("get_latitude_longitude_by_city", description="Get the weather by city")
def get_latitude_longitude_by_city(city: str):
    """Return coordinates by city"""
    import requests
    from urllib.parse import quote
    
    url = f"http://geocoding-api.open-meteo.com/v1/search?name={quote(city)}&count=1&language=es&format=json"
    response = requests.get(url)
    data = response.json()
    
    if "result" not in data or not data["results"]:
        return []
    
    result = data["results"][0]
    latitude = result["latitude"]
    longitude = result["longitud"]
    
    return [latitude,longitude ]


@tool("get_weather", description="Get current weather for a location specified by longitude and latitude")
def get_weather(latitude,longitude):
    """Get weather by specified location"""
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&currrent_weather=true")
    weather = response.json
    
    return weather
    

In [18]:
system_prompt = """
Eres un asistente de ventas que ayuda a encontrar los productos que se necesitan y puedes ayudar a dar el clima de una ciudad

Tus tools son 
- get_latitude_longitude_by_city: para obtener la latitud y longitud de una ciudad
- get_weather: para obtener el clima de una lugar segun su latitud y longitud
- get_products: para obtener los productos que ofrece una tienda

"""

messages = [
    ("system",system_prompt),
    ("user","Dime que productos ofreces en la tienda")
]

tools = [get_products, get_latitude_longitude_by_city, get_weather]
llm_with_tools = llm.bind_tools(tools)
response = llm_with_tools.invoke(messages)
response.tool_calls


[{'name': 'get_products',
  'args': {},
  'id': 'call_A1WyrOGZwClmB5VJAGPTITWx',
  'type': 'tool_call'}]

In [17]:
messages = [
    ("system",system_prompt),
    ("user","Hola, que tal?")
]
response = llm_with_tools.invoke(messages)
response.text

'¡Hola! Estoy aquí para ayudarte. ¿En qué puedo asistirte hoy?'

In [19]:
system_prompt = """
Eres un asistente de ventas que ayuda a encontrar los productos que se necesitan y puedes ayudar a dar el clima de una ciudad

Tus tools son 
- get_latitude_longitude_by_city: para obtener la latitud y longitud de una ciudad
- get_weather: para obtener el clima de una lugar segun su latitud y longitud
- get_products: para obtener los productos que ofrece una tienda

"""

messages = [
    ("system",system_prompt),
    ("user","Cual es el clima en Apizaco")
]

tools = [get_products, get_latitude_longitude_by_city, get_weather]
llm_with_tools = llm.bind_tools(tools)
response = llm_with_tools.invoke(messages)
response.tool_calls


[{'name': 'get_latitude_longitude_by_city',
  'args': {'city': 'Apizaco'},
  'id': 'call_C3bz04mBpv3CQRaO5QHnQbil',
  'type': 'tool_call'}]